# Week 29, Day 6: Forge: Week 29 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w029/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", round(26.725 * np.log(50 / 140), 1), "mV")
print("2.", 50e3 * 1e-6 * 1e3, "ms, whatever the size")
print("3.", round(2 * np.sqrt(1000 * 0.5 * 0.5), 1), "pA")

### Exercise: Part A: running down

Complete `run_down(d_um)`: at each step compute the GHK currents of sodium and potassium through the membrane of a sphere of diameter d_um (A, positive outward), then change the internal concentrations by -I dt / (F vol), converted from mol/cm³ to mM (times 10⁶). Return the time at which the voltage has risen 10 mV above its starting value. Use time steps of 0.025 d_um seconds.

In [ ]:
import numpy as np
F, VT = 96485.0, 26.725
PK = 1e-6; PNa = 0.05 * PK                 # cm/s
Ko, Nao = 5.0, 145.0                       # mM

def voltage(Ki, Nai):
    return VT * np.log((PK * Ko + PNa * Nao) / (PK * Ki + PNa * Nai))

def current(P, V, ci, co):                 # A/cm^2, concentrations in mM
    u = V / VT
    return P * F * u * (ci - co * np.exp(-u)) / (1 - np.exp(-u)) * 1e-6

def run_down(d_um):
    d = d_um * 1e-4; area = np.pi * d ** 2; vol = np.pi * d ** 3 / 6
    Ki, Nai, t, dt = 140.0, 12.0, 0.0, 0.025 * d_um
    v0 = voltage(Ki, Nai)
    while voltage(Ki, Nai) - v0 < 10:
        v = voltage(Ki, Nai)
        I_Na = current(PNa, v, Nai, Nao) * area
        I_K = ...
        Nai += ...
        Ki += ...
        t += dt
    return t

t10 = {d: run_down(d) for d in (20, 1)}
for d, t in t10.items():
    print(f"d = {d:2d} um: 10 mV of depolarisation after {t:6.0f} s ({t / 60:.1f} min)")

In [ ]:
import numpy as np
def test_current():
    "current gives the GHK current density in A/cm^2 for concentrations in mM"
    u = -65 / VT
    assert abs(current(1e-6, -65, 140, 5) - 1e-6 * F * u * (140 - 5 * np.exp(-u)) / (1 - np.exp(-u)) * 1e-6) < 1e-15

def test_times():
    "the 20 um cell takes 10 to 20 minutes to depolarise by 10 mV, and the time scales with diameter"
    assert 600 < t10[20] < 1200 and abs(t10[20] / t10[1] - 20) < 1

test_current()
test_times()
print("All checks passed")

### Exercise: Part B: how fast does sodium build up?

For axons of diameter 0.2 μm, 1 μm and 500 μm (the squid), compute the rise in internal sodium per spike `dNa` (mM), and the number of spikes `n_spikes` that would raise it by 10% of its 12 mM resting value if the pump did nothing in between.

In [ ]:
import numpy as np
c_m, dV, f, F = 1e-6, 0.1, 1.3, 96485.0
d = np.array([0.2e-4, 1e-4, 500e-4])          # cm
dNa = ...
n_spikes = ...
for di, x, n in zip(d * 1e4, dNa, n_spikes):
    print(f"d = {di:6.1f} um: {x:.2e} mM per spike, {n:9.0f} spikes for a 10% rise")

In [ ]:
import numpy as np
def test_load():
    "dNa is c_m dV f / (F d / 4) in mM, for diameters in cm"
    assert np.allclose(dNa, 1e-6 * 0.1 * 1.3 / (96485 * d / 4) * 1e6)

def test_spikes():
    "n_spikes is the number of spikes to raise internal sodium by 10% of 12 mM, and is below 10 for the 0.2 um axon"
    assert np.allclose(n_spikes, 1.2 / dNa) and n_spikes[0] < 10

test_load()
test_spikes()
print("All checks passed")